# Single-Hidden-Layer Regression MLP with Hand-Derived Gradients

This notebook fits a one-hidden-layer neural network for regression using **gradient descent with analytically derived gradients** — no autograd. Each piece of code is preceded by the math it implements.

**Roadmap**
1. Notation and model
2. Activation function and initialization
3. Forward pass
4. Loss function
5. Gradients (the backward pass, derived by hand)
6. Finite-difference gradient check
7. Gradient descent (`fit`)
8. Teacher–student experiment on simulated data

In [1]:
import numpy as np

## 1. Notation and model

**Data**
- $X \in \mathbb{R}^{n \times p}$: design matrix, one row per observation $x_i^\top$
- $y \in \mathbb{R}^{n}$: targets

**Parameters** ($K$ hidden units)
- $W^{[1]} \in \mathbb{R}^{p \times K}$, $b^{[1]} \in \mathbb{R}^{K}$: hidden layer
- $w^{[2]} \in \mathbb{R}^{K}$, $b^{[2]} \in \mathbb{R}$: output layer
- $g$: hidden activation (ReLU)

In ISL's notation, column $k$ of $W^{[1]}$ and $b^{[1]}_k$ are the weights $w_{k1},\dots,w_{kp}$ and bias $w_{k0}$ of hidden unit $A_k$; $w^{[2]}$ and $b^{[2]}$ are $\beta_1,\dots,\beta_K$ and $\beta_0$.

**Shapes in code.** Vectors are stored as 1-D NumPy arrays: `y`, `y_hat` have shape `(n,)`, and `b1`, `w2` have shape `(K,)`. This avoids a silent broadcasting bug: if `y` is `(n,)` and `y_hat` is `(n, 1)`, then `y - y_hat` broadcasts to an `(n, n)` matrix.

| Array | Shape |
|---|---|
| `X` | `(n, p)` |
| `y`, `y_hat` | `(n,)` |
| `W1` | `(p, K)` |
| `b1`, `w2` | `(K,)` |
| `b2` | scalar |
| `Z`, `A` | `(n, K)` |

## 2. Activation function and its derivative

The ReLU activation and its derivative, applied elementwise:

$$
g(z) = \max(0, z), \qquad
g'(z) = \mathbb{1}[z > 0] = \begin{cases} 1 & z > 0 \\ 0 & z \le 0 \end{cases}
$$

ReLU is not differentiable at $z = 0$; we use the common convention $g'(0) = 0$.

In [2]:
def relu(Z):
    """Elementwise ReLU: g(z) = max(0, z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """Elementwise ReLU derivative: g'(z) = 1[z > 0] (0 at z = 0 by convention)."""
    return (Z > 0).astype(float)

## 3. Parameter initialization

Unlike linear regression, the MLP loss is **not convex**, and zero initialization fails:
- **Symmetry:** if all hidden units start identical, they receive identical gradients and stay identical forever.
- **ReLU at zero:** with $W^{[1]} = 0$ and $b^{[1]} = 0$, every $z_{ik} = 0$, so $g'(Z) = 0$ and the hidden-layer gradients vanish.

We use **He initialization** for the hidden weights, which keeps the variance of the pre-activations roughly constant for ReLU units, and a similar scaling for the output weights:

$$
W^{[1]}_{jk} \sim N\!\left(0, \tfrac{2}{p}\right), \qquad
w^{[2]}_k \sim N\!\left(0, \tfrac{1}{K}\right), \qquad
b^{[1]} = 0, \qquad b^{[2]} = 0
$$

Parameters are stored in a dict so that the gradients can use the same keys and shapes.

In [3]:
def init_params(p, K, rng=None):
    """He initialization for the hidden layer; small random output weights.

    Weights are random (not zero) to break symmetry between hidden units.
    """
    rng = np.random.default_rng() if rng is None else rng
    return {
        "W1": rng.normal(0.0, np.sqrt(2.0 / p), size=(p, K)),
        "b1": np.zeros(K),
        "w2": rng.normal(0.0, np.sqrt(1.0 / K), size=K),
        "b2": 0.0,
    }

## 4. Forward pass

**Scalar form** (observation $i$, hidden unit $k$):

$$
z_{ik} = b^{[1]}_k + \sum_{j=1}^{p} x_{ij} W^{[1]}_{jk}, \qquad
a_{ik} = g(z_{ik}), \qquad
\hat y_i = b^{[2]} + \sum_{k=1}^{K} w^{[2]}_k \, a_{ik}
$$

**Matrix form** ($\mathbf{1}$ is a column of $n$ ones):

$$
\begin{aligned}
Z &= XW^{[1]} + \mathbf{1}\, b^{[1]\top} && (n \times K) \quad \text{pre-activations} \\
A &= g(Z) && (n \times K) \quad \text{hidden activations} \\
\hat y &= A\,w^{[2]} + b^{[2]}\mathbf{1} && (n \times 1) \quad \text{predictions}
\end{aligned}
$$

In NumPy, broadcasting `X @ W1 + b1` plays the role of the outer product $\mathbf{1}\, b^{[1]\top}$.

The function also returns a **cache** of $Z$ and $A$, which the backward pass needs: $A$ for the output-layer gradients, and $Z$ for $g'(Z)$.

In [4]:
def forward(X, params):
    """Forward pass. Returns y_hat (n,) and a cache with Z and A, each (n, K)."""
    W1, b1 = params["W1"], params["b1"]
    w2, b2 = params["w2"], params["b2"]

    Z = X @ W1 + b1          # broadcasting adds b1 to every row
    A = relu(Z)
    y_hat = A @ w2 + b2      # (n, K) @ (K,) -> (n,)

    cache = {"Z": Z, "A": A}
    return y_hat, cache

### Check: vectorized forward pass vs. the scalar formulas

We verify the shapes, then recompute $\hat y_i$ with explicit loops over $j$ and $k$ (the scalar formulas above) and confirm they match the matrix version.

In [5]:
rng = np.random.default_rng(0)
n, p, K = 5, 3, 4

X = rng.normal(size=(n, p))
params = init_params(p, K, rng)
y_hat, cache = forward(X, params)

# Shape checks
assert cache["Z"].shape == (n, K)
assert cache["A"].shape == (n, K)
assert y_hat.shape == (n,)

# Cross-check against the scalar formulas, one observation at a time
for i in range(n):
    z_i = [params["b1"][k] + sum(X[i, j] * params["W1"][j, k] for j in range(p))
           for k in range(K)]
    a_i = [max(0.0, z) for z in z_i]
    yhat_i = params["b2"] + sum(params["w2"][k] * a_i[k] for k in range(K))
    assert np.isclose(yhat_i, y_hat[i])

print("y_hat:", y_hat)
print("fraction of active hidden units:", (cache["Z"] > 0).mean())
print("forward pass matches the scalar formulas")

y_hat: [-0.09850686 -0.11709895 -0.103923   -0.26028959 -0.44002394]
fraction of active hidden units: 0.55
forward pass matches the scalar formulas


## 5. Loss function

Mean squared error:

$$
L = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat y_i)^2 = \frac{1}{n}\lVert y - \hat y \rVert^2
$$

`mse` takes predictions directly; `loss` takes the parameters, runs the forward pass, and returns $L(\theta)$. The gradient check needs the second form, since it perturbs parameters.

In [6]:
def mse(y, y_hat):
    """L = (1/n) * ||y - y_hat||^2"""
    return np.mean((y - y_hat) ** 2)


def loss(X, y, params):
    """MSE loss as a function of the parameters (forward pass + MSE)."""
    y_hat, _ = forward(X, params)
    return mse(y, y_hat)

## 6. Gradients (backward pass)

We work backward from the loss, defining an **error signal** at each stage.

**Step 1 — loss with respect to the predictions:**

$$
\frac{\partial L}{\partial \hat y_i} = \frac{2}{n}(\hat y_i - y_i)
\quad\Longrightarrow\quad
\delta := \frac{2}{n}(\hat y - y) \in \mathbb{R}^n
$$

**Step 2 — output-layer parameters.** Since $\partial \hat y_i / \partial w^{[2]}_k = a_{ik}$ and $\partial \hat y_i / \partial b^{[2]} = 1$:

$$
\frac{\partial L}{\partial w^{[2]}_k} = \sum_{i} \delta_i\, a_{ik}
\;\Longrightarrow\;
\boxed{\nabla_{w^{[2]}} L = A^\top \delta}
\qquad\qquad
\boxed{\frac{\partial L}{\partial b^{[2]}} = \mathbf{1}^\top \delta = \sum_i \delta_i}
$$

This is the linear-regression gradient $\frac{2}{n}X^\top(\hat y - y)$ with $X$ replaced by the learned features $A$.

**Step 3 — push the error into the hidden layer.** By the chain rule,

$$
\frac{\partial L}{\partial z_{ik}}
= \underbrace{\delta_i}_{\partial L/\partial \hat y_i}\;
  \underbrace{w^{[2]}_k}_{\partial \hat y_i/\partial a_{ik}}\;
  \underbrace{g'(z_{ik})}_{\partial a_{ik}/\partial z_{ik}}
\quad\Longrightarrow\quad
\boxed{\Delta = \big(\delta\, w^{[2]\top}\big) \odot g'(Z)} \in \mathbb{R}^{n \times K}
$$

where $\delta\, w^{[2]\top}$ is an outer product and $\odot$ is the elementwise (Hadamard) product.

**Step 4 — hidden-layer parameters.** $W^{[1]}_{jk}$ affects only column $k$ of $Z$, with $\partial z_{ik}/\partial W^{[1]}_{jk} = x_{ij}$:

$$
\frac{\partial L}{\partial W^{[1]}_{jk}} = \sum_{i} x_{ij}\, \Delta_{ik}
\;\Longrightarrow\;
\boxed{\nabla_{W^{[1]}} L = X^\top \Delta}
\qquad\qquad
\boxed{\nabla_{b^{[1]}} L = \Delta^\top \mathbf{1}}
$$

**The pattern:** weight gradient = (layer input)$^\top$ × (error signal at that layer's pre-activation); bias gradient = column sums of the error signal.

| Quantity | Formula | Shape |
|---|---|---|
| $\delta$ | $\frac{2}{n}(\hat y - y)$ | $n$ |
| $\nabla_{w^{[2]}} L$ | $A^\top \delta$ | $K$ |
| $\partial L / \partial b^{[2]}$ | $\mathbf{1}^\top \delta$ | scalar |
| $\Delta$ | $(\delta\, w^{[2]\top}) \odot g'(Z)$ | $n \times K$ |
| $\nabla_{W^{[1]}} L$ | $X^\top \Delta$ | $p \times K$ |
| $\nabla_{b^{[1]}} L$ | $\Delta^\top \mathbf{1}$ | $K$ |

In [7]:
def gradients(X, y, y_hat, params, cache):
    """Hand-derived gradients of the MSE loss.

    Returns a dict with the same keys (and shapes) as params.
    """
    n = X.shape[0]
    Z, A = cache["Z"], cache["A"]
    w2 = params["w2"]

    # Step 1: error signal at the output
    delta = (2.0 / n) * (y_hat - y)               # (n,)

    # Step 2: output-layer gradients (linear regression on features A)
    dw2 = A.T @ delta                             # (K, n) @ (n,) -> (K,)
    db2 = delta.sum()

    # Step 3: push the error back through w2 and the ReLU
    Delta = np.outer(delta, w2) * relu_grad(Z)    # (n, K)

    # Step 4: hidden-layer gradients
    dW1 = X.T @ Delta                             # (p, n) @ (n, K) -> (p, K)
    db1 = Delta.sum(axis=0)                       # (K,)

    return {"W1": dW1, "b1": db1, "w2": dw2, "b2": db2}

### Check: every gradient has the same shape as its parameter

In [8]:
y = rng.normal(size=n)
grads = gradients(X, y, y_hat, params, cache)
for k in params:
    assert np.shape(grads[k]) == np.shape(params[k]), k
print("gradient shapes:", {k: np.shape(v) for k, v in grads.items()})

gradient shapes: {'W1': (3, 4), 'b1': (4,), 'w2': (4,), 'b2': ()}


## 7. Finite-difference gradient check

For each scalar parameter $\theta_j$, approximate the partial derivative with a **central difference**:

$$
\frac{\partial L}{\partial \theta_j} \approx
\frac{L(\theta + \epsilon e_j) - L(\theta - \epsilon e_j)}{2\epsilon}
$$

where $e_j$ is the $j$-th standard basis vector. A Taylor expansion shows the error is $O(\epsilon^2)$, versus $O(\epsilon)$ for a one-sided difference.

For each parameter block we report the **relative error**

$$
\text{rel. error} = \frac{\lVert g_{\text{num}} - g_{\text{ana}} \rVert}{\lVert g_{\text{num}} \rVert + \lVert g_{\text{ana}} \rVert}
$$

which is scale-free, so one tolerance works for large and small gradients. In float64 with $\epsilon = 10^{-6}$: about $10^{-9}$ or smaller means correct; $10^{-2}$ or larger means a bug.

**ReLU kink:** $g$ is not differentiable at $0$. If some $z_{ik}$ lies within $\epsilon$ of zero, the finite difference straddles the kink and that entry can disagree even though the analytic gradient is correct. This is rare with random data; re-run with a different seed if a single entry is off.

**Cost:** two forward passes per scalar parameter, so use it on small problems to validate code, not during training.

In [9]:
def grad_check(X, y, params, eps=1e-6, tol=1e-7, verbose=True):
    """Compare analytic gradients to central finite differences.

    Returns a dict {param_name: relative_error} and whether all passed.
    """
    y_hat, cache = forward(X, params)
    analytic = gradients(X, y, y_hat, params, cache)

    errors = {}
    for name, value in params.items():
        theta = np.array(value, dtype=float)       # copy; 0-d array for b2
        numerical = np.zeros_like(theta)

        for idx in np.ndindex(theta.shape):        # yields () once for a scalar
            trial = dict(params)                   # shallow copy: swap one entry
            original = theta[idx]

            theta[idx] = original + eps
            trial[name] = theta
            L_plus = loss(X, y, trial)

            theta[idx] = original - eps
            L_minus = loss(X, y, trial)

            theta[idx] = original                  # restore
            numerical[idx] = (L_plus - L_minus) / (2.0 * eps)

        ana = np.asarray(analytic[name], dtype=float)
        num_diff = np.linalg.norm(numerical - ana)
        denom = max(np.linalg.norm(numerical) + np.linalg.norm(ana), 1e-12)
        errors[name] = num_diff / denom

    passed = all(err < tol for err in errors.values())
    if verbose:
        for name, err in errors.items():
            flag = "ok" if err < tol else "FAIL"
            print(f"  {name:>3}: relative error = {err:.2e}  [{flag}]")
    return errors, passed

### Run the gradient check

We use a slightly larger problem and set nonzero biases so every parameter is exercised.

In [10]:
n, p, K = 50, 3, 6
X = rng.normal(size=(n, p))
y = rng.normal(size=n)
params = init_params(p, K, rng)
params["b1"] = rng.normal(0.0, 0.1, size=K)
params["b2"] = 0.3

print("gradient check:")
_, passed = grad_check(X, y, params)
assert passed, "gradient check failed"

gradient check:
   W1: relative error = 2.72e-10  [ok]
   b1: relative error = 7.25e-11  [ok]
   w2: relative error = 3.68e-11  [ok]
   b2: relative error = 1.15e-11  [ok]


### Sanity check: does the gradient check catch a bug?

To make sure the check is not passing trivially, plant a 1% error in $\nabla_{W^{[1]}} L$. Only `W1` should fail.

In [11]:
_correct_gradients = gradients

def gradients_buggy(X, y, y_hat, params, cache):
    g = _correct_gradients(X, y, y_hat, params, cache)
    g["W1"] = 1.01 * g["W1"]                      # deliberate 1% error
    return g

gradients = gradients_buggy                      # grad_check looks up `gradients` at call time
try:
    print("gradient check with a planted bug:")
    _, passed = grad_check(X, y, params)
finally:
    gradients = _correct_gradients                # always restore the correct version
print("passed:", passed)

gradient check with a planted bug:
   W1: relative error = 4.98e-03  [FAIL]
   b1: relative error = 7.25e-11  [ok]
   w2: relative error = 3.68e-11  [ok]
   b2: relative error = 1.15e-11  [ok]
passed: False


## 8. Gradient descent

With learning rate $\eta$, each iteration updates every parameter $\theta \in \{W^{[1]}, b^{[1]}, w^{[2]}, b^{[2]}\}$ by

$$
\theta^{(t+1)} = \theta^{(t)} - \eta\, \nabla_\theta L\big(\theta^{(t)}\big)
$$

Each iteration of full-batch gradient descent:
1. **Forward pass** → $\hat y$, cache $(Z, A)$
2. **Record the loss** $L(\theta^{(t)})$ (free, since $\hat y$ is already computed)
3. **Backward pass** → gradients, same keys and shapes as the parameters
4. **Update** every parameter

`fit` also stops with an error if the loss becomes non-finite (learning rate too large), and logs the number of **dead units** — hidden units with $z_{ik} \le 0$ for every observation $i$. A dead unit's column of $\Delta$ is all zeros, so it never receives a gradient and cannot recover.

In [12]:
def predict(X, params):
    """Predictions only (no cache)."""
    y_hat, _ = forward(X, params)
    return y_hat


def fit(X, y, K, lr=0.05, n_iters=5000, params=None, rng=None,
        log_every=500, verbose=True):
    """Full-batch gradient descent on the MSE loss.

    Args:
        X, y      : training data, shapes (n, p) and (n,)
        K         : number of hidden units (ignored if params is given)
        lr        : learning rate eta
        n_iters   : number of gradient-descent steps
        params    : optional starting parameters (e.g. to resume training);
                    if None, uses init_params(p, K, rng)
        rng       : numpy Generator for reproducible initialization
        log_every : print the loss every this many iterations

    Returns:
        params  : fitted parameters (a new dict; the input is not modified)
        history : array of training losses, one per iteration
    """
    y = np.asarray(y, dtype=float).ravel()     # guard against (n, 1) targets
    n, p = X.shape
    if params is None:
        params = init_params(p, K, rng)
    else:
        params = {k: np.array(v, dtype=float) for k, v in params.items()}

    history = np.empty(n_iters)
    for t in range(n_iters):
        y_hat, cache = forward(X, params)
        history[t] = mse(y, y_hat)

        if not np.isfinite(history[t]):
            raise FloatingPointError(
                f"loss became {history[t]} at iteration {t}; try a smaller lr")

        grads = gradients(X, y, y_hat, params, cache)
        for k in params:
            params[k] = params[k] - lr * grads[k]

        if verbose and (t % log_every == 0 or t == n_iters - 1):
            dead = np.all(cache["Z"] <= 0, axis=0).sum()
            print(f"  iter {t:>6}: loss = {history[t]:.5f}   dead units = {dead}/{params['w2'].size}")

    return params, history

## 9. Teacher–student experiment

Simulate data from a **known random network** (the teacher) with $p = 3$ inputs and $K_{\text{true}} = 4$ hidden units, plus Gaussian noise:

$$
y_i = f_{\text{teacher}}(x_i) + \varepsilon_i, \qquad
x_i \sim N(0, I_p), \qquad
\varepsilon_i \sim N(0, \sigma^2), \quad \sigma = 0.1
$$

Since the noise is independent of $x$, the best achievable expected test MSE is the noise variance $\sigma^2 = 0.01$ (the irreducible error).

In [13]:
rng = np.random.default_rng(42)
p, K_true, sigma = 3, 4, 0.1

teacher = init_params(p, K_true, rng)
teacher["b1"] = rng.normal(0.0, 0.5, size=K_true)
teacher["w2"] = rng.normal(0.0, 1.0, size=K_true)
teacher["b2"] = 0.5

def simulate(n):
    X = rng.normal(size=(n, p))
    y = predict(X, teacher) + sigma * rng.normal(size=n)
    return X, y

X_train, y_train = simulate(1000)
X_test, y_test = simulate(5000)
print("train:", X_train.shape, y_train.shape, "  test:", X_test.shape, y_test.shape)

train: (1000, 3) (1000,)   test: (5000, 3) (5000,)


### Fit a student network

The student is wider than the teacher ($K = 8$), trained with full-batch gradient descent, $\eta = 0.1$, for 5,000 iterations.

In [14]:
params, history = fit(X_train, y_train, K=8, lr=0.1, n_iters=5000,
                      rng=np.random.default_rng(0))

  iter      0: loss = 3.11469   dead units = 0/8
  iter    500: loss = 0.01226   dead units = 0/8
  iter   1000: loss = 0.01018   dead units = 0/8
  iter   1500: loss = 0.00973   dead units = 0/8
  iter   2000: loss = 0.00949   dead units = 0/8


  iter   2500: loss = 0.00937   dead units = 0/8


  iter   3000: loss = 0.00929   dead units = 0/8


  iter   3500: loss = 0.00926   dead units = 0/8
  iter   4000: loss = 0.00925   dead units = 0/8
  iter   4500: loss = 0.00924   dead units = 0/8
  iter   4999: loss = 0.00924   dead units = 0/8


### Compare to the noise floor and to linear regression

As a baseline, fit ordinary least squares on the same features:

$$
\hat\beta = \arg\min_\beta \lVert y - \tilde X \beta \rVert^2, \qquad \tilde X = [\mathbf{1} \;\; X]
$$

Linear regression cannot represent the teacher's piecewise-linear function, so it should do much worse than the MLP.

In [15]:
X1 = np.column_stack([np.ones(len(X_train)), X_train])
beta = np.linalg.lstsq(X1, y_train, rcond=None)[0]
ols_test = mse(y_test, np.column_stack([np.ones(len(X_test)), X_test]) @ beta)

print(f"noise variance (best achievable MSE): {sigma**2:.5f}")
print(f"MLP  train MSE: {history[-1]:.5f}   test MSE: {mse(y_test, predict(X_test, params)):.5f}")
print(f"OLS  test MSE: {ols_test:.5f}")

noise variance (best achievable MSE): 0.01000
MLP  train MSE: 0.00924   test MSE: 0.01041
OLS  test MSE: 0.10798


### Why compare predictions, not weights

Even with $K = K_{\text{true}}$, the fitted weights would generally not match the teacher's, because the network function is invariant to two kinds of reparameterization:

- **Permutation:** reordering the hidden units (columns of $W^{[1]}$, entries of $b^{[1]}$ and $w^{[2]}$) gives the same function.
- **Positive scaling:** ReLU is positively homogeneous, $g(cz) = c\,g(z)$ for $c > 0$. Scaling unit $k$'s incoming weights and bias by $c$ and its outgoing weight by $1/c$ leaves $\hat y$ unchanged:

$$
\frac{w^{[2]}_k}{c}\, g\big(c\,(b^{[1]}_k + x^\top W^{[1]}_{\cdot k})\big) = w^{[2]}_k\, g\big(b^{[1]}_k + x^\top W^{[1]}_{\cdot k}\big)
$$

So we judge the fit by test-set predictions, not by parameter recovery.